In [4]:
#Kudaibergenova Aiym
!pip install datasets torch gensim scikit-learn

In [2]:
# Импорт всех необходимых библиотек
import re
import random
import numpy as np
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_sequence

from datasets import load_dataset
from sklearn.linear_model import LogisticRegression
import gensim
from gensim.models import Word2Vec

import warnings
warnings.filterwarnings('ignore')

# Фиксируем seed для воспроизводимости результатов
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# Определяем устройство: GPU если доступен, иначе CPU
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Устройство: {DEVICE}')

Устройство: cpu


In [3]:
# Part A — Загружаем датасет IMDB
dataset  = load_dataset('imdb')
train_raw = dataset['train']
test_raw  = dataset['test']

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [5]:
# Part A — Делим официальный train на train (80%) и validation (20%)
all_train = list(zip(train_raw['text'], train_raw['label']))
random.shuffle(all_train)

split     = int(0.8 * len(all_train))
train_set = all_train[:split]
val_set   = all_train[split:]
test_set  = list(zip(test_raw['text'], test_raw['label']))

train_texts, train_labels = zip(*train_set)
val_texts,   val_labels   = zip(*val_set)
test_texts,  test_labels  = zip(*test_set)

train_labels = list(train_labels)
val_labels   = list(val_labels)
test_labels  = list(test_labels)

In [6]:
# Part A — Выводим размеры сплитов, баланс классов и среднюю длину отзыва
print('Размеры сплитов:')
print(f'  Train : {len(train_texts):,}')
print(f'  Val   : {len(val_texts):,}')
print(f'  Test  : {len(test_texts):,}')

pos = sum(train_labels)
neg = len(train_labels) - pos
print(f'\nБаланс классов (train): {pos:,} позитивных ({pos/len(train_labels):.1%}), '
      f'{neg:,} негативных ({neg/len(train_labels):.1%})')

avg_chars = np.mean([len(t) for t in train_texts])
avg_words = np.mean([len(t.split()) for t in train_texts])
print(f'\nСредняя длина отзыва: {avg_chars:.0f} символов / {avg_words:.0f} слов')

Размеры сплитов:
  Train : 20,000
  Val   : 5,000
  Test  : 25,000

Баланс классов (train): 9,979 позитивных (49.9%), 10,021 негативных (50.1%)

Средняя длина отзыва: 1326 символов / 234 слов


In [7]:
# Part A — Показываем 2 позитивных и 2 негативных примера из train
pos_examples = [(t, l) for t, l in zip(train_texts, train_labels) if l == 1][:2]
neg_examples = [(t, l) for t, l in zip(train_texts, train_labels) if l == 0][:2]

print('=' * 70)
print('2 ПОЗИТИВНЫХ примера (label = 1):')
print('=' * 70)
for i, (text, _) in enumerate(pos_examples):
    print(f'\n[{i+1}] {text[:400]}...')

print('\n' + '=' * 70)
print('2 НЕГАТИВНЫХ примера (label = 0):')
print('=' * 70)
for i, (text, _) in enumerate(neg_examples):
    print(f'\n[{i+1}] {text[:400]}...')

2 ПОЗИТИВНЫХ примера (label = 1):

[1] Eric Clapton, Jack Bruce and Ginger Baker re-unite to play all their songs from 35 years ago when they formed a trio called "Cream." Those were the psychedelic days of England and America and these guys looked it: all skinny, very long hair, wild clothes and loud music. They played a combination of rock and blues and it was, for the most part, good stuff.<br /><br />Well, these guys are now 60-som...

[2] This film caught me off guard when it started out in a Cafe located in Arizona and a Richard Grieco,(Rex),"Dead Easy",'04, decides to have something to eat and gets all hot and bothered over a very hot, sexy waitress. While Rex steps out of the Cafe, he sees a State Trooper and asks him,"ARE YOU FAST?" and then all hell breaks loose in more ways than one. Nancy Allen (Maggie Hewitt),"Dressed to Ki...

2 НЕГАТИВНЫХ примера (label = 0):

[1] You have to understand, when Wargames was released in 1983, it created a generation of wannabe computer hack

In [8]:
# Part A — Вариант препроцессинга 1: простой
# Шаги: lowercase → убираем HTML теги → убираем пунктуацию → split по пробелу

def preprocess_simple(text):
    text = text.lower()                            # приводим к нижнему регистру
    text = re.sub(r'<br\s*/?>', ' ', text)         # убираем HTML теги <br>
    text = re.sub(r'[^a-z0-9\s]', ' ', text)      # убираем пунктуацию
    text = re.sub(r'\s+', ' ', text).strip()       # убираем лишние пробелы
    return text.split()

In [9]:
# Part A — Вариант препроцессинга 2: расширенный
# Шаги: всё то же самое + удаление стоп-слов (НО сохраняем отрицания: not, no, never)

STOPWORDS = {
    'the','a','an','is','it','this','that','was','are','be','for','on',
    'with','as','at','by','from','or','have','had','to','of','in','and',
    'do','did','does','so','if','which','when','there','than','then',
    'were','been','will','would','could','should','all','any','what',
    'who','how','up','out','some','he','she','they','we','you','my','its',
    'me','him','her','us','them','our','your','their','am','been','being'
    # 'not', 'no', 'never' намеренно НЕ включены — важны для сентимента
}

def preprocess_advanced(text):
    text = text.lower()
    text = re.sub(r'<br\s*/?>', ' ', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = text.split()
    # Убираем стоп-слова и однобуквенные токены
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 1]
    return tokens

In [10]:
# Part A — Применяем оба варианта к train и выводим статистику
# Для каждого варианта: 5 токенизированных примеров, размер словаря, топ-10, 10 редких

def report_preprocessing(name, fn, texts):
    tokenized = [fn(t) for t in texts]
    vocab     = Counter(tok for toks in tokenized for tok in toks)

    print(f'\n{"-"*65}')
    print(f'Вариант: {name}')
    print(f'{"-"*65}')
    print(f'Размер словаря (до обрезки): {len(vocab):,}')
    print(f'10 самых частых токенов:')
    print(f'  {[w for w, _ in vocab.most_common(10)]}')
    rare = [w for w, c in vocab.items() if c == 1][:10]
    print(f'10 редких токенов (freq=1):')
    print(f'  {rare}')
    print(f'5 токенизированных примеров:')
    for i, toks in enumerate(tokenized[:5]):
        print(f'  [{i+1}] {toks[:15]}')
    return tokenized, vocab

train_simple,   vocab_simple   = report_preprocessing('Simple  (lower + punct + split)',   preprocess_simple,   train_texts)
train_advanced, vocab_advanced = report_preprocessing('Advanced (+ удаление стоп-слов)',   preprocess_advanced, train_texts)


-----------------------------------------------------------------
Вариант: Simple  (lower + punct + split)
-----------------------------------------------------------------
Размер словаря (до обрезки): 68,112
10 самых частых токенов:
  ['the', 'and', 'a', 'of', 'to', 'is', 'it', 'in', 'i', 'this']
10 редких токенов (freq=1):
  ['bioweapons', 'gulled', 'snt', 'synthesize', 'fruedian', 'bonsais', 'aguilera', 'leut', 'embeciles', 'allegorically']
5 токенизированных примеров:
  [1] ['eric', 'clapton', 'jack', 'bruce', 'and', 'ginger', 'baker', 're', 'unite', 'to', 'play', 'all', 'their', 'songs', 'from']
  [2] ['you', 'have', 'to', 'understand', 'when', 'wargames', 'was', 'released', 'in', '1983', 'it', 'created', 'a', 'generation', 'of']
  [3] ['this', 'movie', 'purports', 'to', 'be', 'a', 'character', 'study', 'of', 'perversion', 'some', 'reviewers', 'have', 'been', 'gulled']
  [4] ['this', 'film', 'caught', 'me', 'off', 'guard', 'when', 'it', 'started', 'out', 'in', 'a', 'cafe', 'locat

### Part A — Вывод по препроцессингу

**Lowercase** уменьшает словарь без потери смысла — «Good» и «good» становятся одним токеном. **Удаление HTML-тегов** (`<br/>`) однозначно полезно — они не несут смысла. **Удаление пунктуации** убирает шум. **Удаление стоп-слов** уменьшает словарь, но опасно для сентимент-анализа: слова *«not»*, *«no»*, *«never»* являются стоп-словами, но полностью меняют смысл фразы («I do **not** like this» → без «not» → «I like this»). Простой вариант безопаснее для задачи сентимента.

---
## Part B: Word-Level Integer Encoding Model

In [11]:
# Part B — Гиперпараметры модели
MIN_FREQ    = 3    # минимальная частота слова для попадания в словарь
MAX_SEQ_LEN = 256  # максимальная длина последовательности (токенов)
EMBED_DIM   = 128  # размерность эмбеддинга
HIDDEN_SIZE = 128  # размер скрытого состояния GRU
BATCH_SIZE  = 64
EPOCHS_B    = 5
LR          = 1e-3

In [12]:
# Part B — Строим словарь ТОЛЬКО из тренировочных данных
# Специальные токены: <PAD> для паддинга, <UNK> для неизвестных слов
vocab_counter = Counter(tok for toks in train_simple for tok in toks)

word2idx = {'<PAD>': 0, '<UNK>': 1}
for word, freq in vocab_counter.items():
    if freq >= MIN_FREQ:  # берём только слова с частотой >= MIN_FREQ
        word2idx[word] = len(word2idx)

idx2word   = {v: k for k, v in word2idx.items()}
VOCAB_SIZE = len(word2idx)
PAD_IDX    = 0
UNK_IDX    = 1

print(f'min_freq             : {MIN_FREQ}')
print(f'Максимальная длина   : {MAX_SEQ_LEN}')
print(f'Размер словаря       : {VOCAB_SIZE:,}')
print(f'Паддинг              : до {MAX_SEQ_LEN} токенов, <PAD> = 0')
print(f'Обрезка              : справа, если длина > {MAX_SEQ_LEN}')

min_freq             : 3
Максимальная длина   : 256
Размер словаря       : 34,479
Паддинг              : до 256 токенов, <PAD> = 0
Обрезка              : справа, если длина > 256


In [13]:
# Part B — Dataset и DataLoader

def encode_text(tokens, word2idx, max_len):
    # Переводим токены в индексы, неизвестные → UNK, обрезаем справа
    return [word2idx.get(t, UNK_IDX) for t in tokens[:max_len]]


class IMDBDataset(Dataset):
    def __init__(self, tokenized_texts, labels, word2idx, max_len):
        self.labels  = labels
        self.encoded = [encode_text(toks, word2idx, max_len) or [UNK_IDX]
                        for toks in tokenized_texts]

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        ids = self.encoded[idx]
        return (torch.tensor(ids,              dtype=torch.long),
                torch.tensor(len(ids),         dtype=torch.long),
                torch.tensor(self.labels[idx], dtype=torch.long))


def collate_fn(batch):
    # Паддим последовательности до одной длины внутри батча
    seqs, lengths, labels = zip(*batch)
    seqs_padded = pad_sequence(seqs, batch_first=True, padding_value=PAD_IDX)
    if seqs_padded.shape[1] > MAX_SEQ_LEN:
        seqs_padded = seqs_padded[:, :MAX_SEQ_LEN]
    lengths = torch.clamp(torch.stack(lengths), max=MAX_SEQ_LEN)
    return seqs_padded, lengths, torch.stack(labels)


# Токенизируем val и test простым препроцессингом
val_simple  = [preprocess_simple(t) for t in val_texts]
test_simple = [preprocess_simple(t) for t in test_texts]

train_ds_b = IMDBDataset(train_simple, train_labels, word2idx, MAX_SEQ_LEN)
val_ds_b   = IMDBDataset(val_simple,   val_labels,   word2idx, MAX_SEQ_LEN)
test_ds_b  = IMDBDataset(test_simple,  test_labels,  word2idx, MAX_SEQ_LEN)

train_loader_b = DataLoader(train_ds_b, BATCH_SIZE, shuffle=True,  collate_fn=collate_fn)
val_loader_b   = DataLoader(val_ds_b,   BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader_b  = DataLoader(test_ds_b,  BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f'Train батчей: {len(train_loader_b)} | Val батчей: {len(val_loader_b)} | Test батчей: {len(test_loader_b)}')

Train батчей: 313 | Val батчей: 79 | Test батчей: 391


In [14]:
# Part B — Архитектура модели: token ids → Embedding → GRU → Linear

class WordClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, num_classes, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.rnn = nn.GRU(embed_dim, hidden_size, batch_first=True)
        self.fc  = nn.Linear(hidden_size, num_classes)

    def forward(self, x, lengths):
        embedded = self.embedding(x)  # [B, T] → [B, T, embed_dim]
        packed   = pack_padded_sequence(embedded, lengths.cpu(),
                                        batch_first=True, enforce_sorted=False)
        _, h_n   = self.rnn(packed)   # берём последнее скрытое состояние
        return self.fc(h_n[-1])       # [B, num_classes]


model_b   = WordClassifier(VOCAB_SIZE, EMBED_DIM, HIDDEN_SIZE, 2, PAD_IDX).to(DEVICE)
optimizer = torch.optim.Adam(model_b.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

print(model_b)
print(f'\nПараметров всего: {sum(p.numel() for p in model_b.parameters()):,}')

WordClassifier(
  (embedding): Embedding(34479, 128, padding_idx=0)
  (rnn): GRU(128, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=2, bias=True)
)

Параметров всего: 4,512,642


In [15]:
# Part B — Вспомогательные функции обучения и оценки

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for seqs, lengths, labels in loader:
        seqs, lengths, labels = seqs.to(DEVICE), lengths.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out  = model(seqs, lengths)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
        correct    += (out.argmax(1) == labels).sum().item()
        total      += len(labels)
    return total_loss / total, correct / total


def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    all_preds, all_true = [], []
    with torch.no_grad():
        for seqs, lengths, labels in loader:
            seqs, lengths, labels = seqs.to(DEVICE), lengths.to(DEVICE), labels.to(DEVICE)
            preds = model(seqs, lengths).argmax(1)
            correct += (preds == labels).sum().item()
            total   += len(labels)
            all_preds.extend(preds.cpu().tolist())
            all_true.extend(labels.cpu().tolist())
    return correct / total, all_preds, all_true

In [17]:
# Part B — Обучаем GRU модель
print('Обучение Word-Level GRU (Part B)...')
for epoch in range(EPOCHS_B):
    tr_loss, tr_acc = train_epoch(model_b, train_loader_b, optimizer, criterion)
    val_acc, _, _   = evaluate(model_b, val_loader_b)
    print(f'  Эпоха {epoch+1}/{EPOCHS_B} | Loss: {tr_loss:.4f} | Train: {tr_acc:.4f} | Val: {val_acc:.4f}')

Обучение Word-Level GRU (Part B)...
  Эпоха 1/5 | Loss: 0.2600 | Train: 0.8995 | Val: 0.8488
  Эпоха 2/5 | Loss: 0.1719 | Train: 0.9387 | Val: 0.8626
  Эпоха 3/5 | Loss: 0.1126 | Train: 0.9614 | Val: 0.8616
  Эпоха 4/5 | Loss: 0.0669 | Train: 0.9791 | Val: 0.8638
  Эпоха 5/5 | Loss: 0.0314 | Train: 0.9908 | Val: 0.8524


In [18]:
# Part B — Итоговые метрики и примеры предсказаний
train_acc_b, _,            _            = evaluate(model_b, train_loader_b)
val_acc_b,   _,            _            = evaluate(model_b, val_loader_b)
test_acc_b,  test_preds_b, test_true_b  = evaluate(model_b, test_loader_b)

print(f'Train Accuracy : {train_acc_b:.4f}')
print(f'Val   Accuracy : {val_acc_b:.4f}')
print(f'Test  Accuracy : {test_acc_b:.4f}')

lbl = lambda x: 'POSITIVE' if x == 1 else 'NEGATIVE'

# 3 верных предсказания
correct_idx   = [i for i,(p,t) in enumerate(zip(test_preds_b, test_true_b)) if p == t][:3]
print('\n3 ВЕРНЫХ предсказания:')
for i in correct_idx:
    print(f'\n  Текст  : {test_texts[i][:250]}...')
    print(f'  Предсказано: {lbl(test_preds_b[i])} | Правда: {lbl(test_true_b[i])}')

# 3 неверных предсказания
incorrect_idx = [i for i,(p,t) in enumerate(zip(test_preds_b, test_true_b)) if p != t][:3]
print('\n3 НЕВЕРНЫХ предсказания:')
for i in incorrect_idx:
    print(f'\n  Текст  : {test_texts[i][:250]}...')
    print(f'  Предсказано: {lbl(test_preds_b[i])} | Правда: {lbl(test_true_b[i])}')

Train Accuracy : 0.9940
Val   Accuracy : 0.8524
Test  Accuracy : 0.8391

3 ВЕРНЫХ предсказания:

  Текст  : I love sci-fi and am willing to put up with a lot. Sci-fi movies/TV are usually underfunded, under-appreciated and misunderstood. I tried to like this, I really did, but it is to good TV sci-fi as Babylon 5 is to Star Trek (the original). Silly prost...
  Предсказано: NEGATIVE | Правда: NEGATIVE

  Текст  : Worth the entertainment value of a rental, especially if you like action movies. This one features the usual car chases, fights with the great Van Damme kick style, shooting battles with the 40 shell load shotgun, and even terrorist style bombs. All ...
  Предсказано: NEGATIVE | Правда: NEGATIVE

  Текст  : its a totally average film with a few semi-alright action sequences that make the plot seem a little better and remind the viewer of the classic van dam films. parts of the plot don't make sense and seem to be added in to use up time. the end plot is...
  Предсказано: NEGAT

In [19]:
# Part C — Обучаем Word2Vec на тренировочных отзывах
print('Обучение Word2Vec...')
w2v = Word2Vec(
    sentences=train_simple,  # токенизированные тренировочные тексты
    vector_size=100,          # размерность вектора
    window=5,                 # размер контекстного окна
    min_count=2,              # минимальная частота слова
    workers=4,
    epochs=5
)
print(f'Словарь Word2Vec : {len(w2v.wv):,}')
print(f'Размерность      : {w2v.vector_size}')

EMBED_SIZE_W2V = w2v.vector_size
RANDOM_UNK     = np.random.randn(EMBED_SIZE_W2V) * 0.01  # вектор для OOV при стратегии random

Обучение Word2Vec...
Словарь Word2Vec : 42,939
Размерность      : 100


In [20]:
# Part C — Функция: отзыв → один вектор (mean pooling)
# Поддерживаем несколько OOV стратегий

def review_to_vector(tokens, w2v_model, oov='zero'):
    """
    oov стратегии:
      'zero'      — OOV слова → нулевой вектор
      'skip'      — OOV слова пропускаем
      'random'    — OOV слова → один общий случайный вектор
      'avg_known' — OOV слова → среднее известных векторов
    """
    vecs, n_found, n_miss = [], 0, 0
    known = [w2v_model.wv[t] for t in tokens if t in w2v_model.wv]

    for tok in tokens:
        if tok in w2v_model.wv:
            vecs.append(w2v_model.wv[tok])
            n_found += 1
        else:
            n_miss += 1
            if   oov == 'zero':      vecs.append(np.zeros(EMBED_SIZE_W2V))
            elif oov == 'random':    vecs.append(RANDOM_UNK)
            elif oov == 'avg_known': vecs.append(np.mean(known, axis=0) if known else np.zeros(EMBED_SIZE_W2V))
            # 'skip' — просто не добавляем

    if not vecs:
        return np.zeros(EMBED_SIZE_W2V), n_found, n_miss
    return np.mean(vecs, axis=0), n_found, n_miss


def build_matrix(tokenized, w2v_model, oov='zero'):
    # Строим матрицу признаков: каждый отзыв → один вектор
    rows, found, miss = [], 0, 0
    for toks in tokenized:
        v, f, m = review_to_vector(toks, w2v_model, oov)
        rows.append(v)
        found += f; miss += m
    return np.array(rows), found, miss

In [21]:
# Part C — Сравниваем 2 стратегии обработки OOV слов
val_simple_c  = [preprocess_simple(t) for t in val_texts]
test_simple_c = [preprocess_simple(t) for t in test_texts]

results_c = {}

for oov_strat in ['zero', 'skip']:
    X_tr, tf, tm = build_matrix(train_simple,   w2v, oov_strat)
    X_va, _,  _  = build_matrix(val_simple_c,   w2v, oov_strat)
    X_te, _,  _  = build_matrix(test_simple_c,  w2v, oov_strat)

    # Классификатор: Logistic Regression на усреднённых векторах
    clf = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
    clf.fit(X_tr, train_labels)

    tr_acc  = clf.score(X_tr, train_labels)
    val_acc = clf.score(X_va, val_labels)
    te_acc  = clf.score(X_te, test_labels)
    results_c[oov_strat] = (tr_acc, val_acc, te_acc)

    print(f'\nOOV стратегия: "{oov_strat}"')
    print(f'  Найдено токенов    : {tf:,}')
    print(f'  Пропущено (OOV)    : {tm:,}')
    print(f'  Train Acc          : {tr_acc:.4f}')
    print(f'  Val   Acc          : {val_acc:.4f}')
    print(f'  Test  Acc          : {te_acc:.4f}')

# Сохраняем лучшую стратегию для итоговой таблицы
best_oov = max(results_c, key=lambda k: results_c[k][2])
train_acc_c, val_acc_c, test_acc_c = results_c[best_oov]
print(f'\nЛучшая OOV стратегия: "{best_oov}" → Test Acc = {test_acc_c:.4f}')


OOV стратегия: "zero"
  Найдено токенов    : 4,743,351
  Пропущено (OOV)    : 25,173
  Train Acc          : 0.8366
  Val   Acc          : 0.8332
  Test  Acc          : 0.8301

OOV стратегия: "skip"
  Найдено токенов    : 4,743,351
  Пропущено (OOV)    : 25,173
  Train Acc          : 0.8363
  Val   Acc          : 0.8326
  Test  Acc          : 0.8310

Лучшая OOV стратегия: "skip" → Test Acc = 0.8310


In [27]:
# Part D — Строим символьный словарь ТОЛЬКО из тренировочных данных
MAX_CHAR_LEN = 512
CHAR_EMBED   = 64
CHAR_HIDDEN  = 64
EPOCHS_D     = 5

all_chars = Counter(ch for text in train_texts for ch in text)
char2idx  = {'<PAD>': 0, '<UNK>': 1}
for ch in all_chars:
    char2idx[ch] = len(char2idx)

CHAR_VOCAB_SIZE = len(char2idx)
print(f'Размер символьного словаря : {CHAR_VOCAB_SIZE}')
print(f'Максимальная длина (символы): {MAX_CHAR_LEN}')

Размер символьного словаря : 170
Максимальная длина (символы): 512


In [23]:
# Part D — Dataset, DataLoader и модель для символьного классификатора

class CharDataset(Dataset):
    def __init__(self, texts, labels, char2idx, max_len):
        self.labels  = labels
        # Каждый символ → индекс, неизвестные → 1 (<UNK>), обрезаем до max_len
        self.encoded = [[char2idx.get(c, 1) for c in t[:max_len]] or [1]
                        for t in texts]

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        ids = self.encoded[idx]
        return (torch.tensor(ids,              dtype=torch.long),
                torch.tensor(len(ids),         dtype=torch.long),
                torch.tensor(self.labels[idx], dtype=torch.long))


def collate_chars(batch):
    seqs, lengths, labels = zip(*batch)
    seqs_padded = pad_sequence(seqs, batch_first=True, padding_value=0)
    if seqs_padded.shape[1] > MAX_CHAR_LEN:
        seqs_padded = seqs_padded[:, :MAX_CHAR_LEN]
    lengths = torch.clamp(torch.stack(lengths), max=MAX_CHAR_LEN)
    return seqs_padded, lengths, torch.stack(labels)


# Архитектура: char ids → Embedding → GRU → Linear
class CharClassifier(nn.Module):
    def __init__(self, char_vocab, embed_dim, hidden, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(char_vocab, embed_dim, padding_idx=0)
        self.rnn = nn.GRU(embed_dim, hidden, batch_first=True)
        self.fc  = nn.Linear(hidden, num_classes)

    def forward(self, x, lengths):
        emb    = self.embedding(x)
        packed = pack_padded_sequence(emb, lengths.cpu(),
                                      batch_first=True, enforce_sorted=False)
        _, h_n = self.rnn(packed)
        return self.fc(h_n[-1])


train_ds_d = CharDataset(train_texts, train_labels, char2idx, MAX_CHAR_LEN)
val_ds_d   = CharDataset(val_texts,   val_labels,   char2idx, MAX_CHAR_LEN)
test_ds_d  = CharDataset(test_texts,  test_labels,  char2idx, MAX_CHAR_LEN)

train_loader_d = DataLoader(train_ds_d, BATCH_SIZE, shuffle=True,  collate_fn=collate_chars)
val_loader_d   = DataLoader(val_ds_d,   BATCH_SIZE, shuffle=False, collate_fn=collate_chars)
test_loader_d  = DataLoader(test_ds_d,  BATCH_SIZE, shuffle=False, collate_fn=collate_chars)

model_d = CharClassifier(CHAR_VOCAB_SIZE, CHAR_EMBED, CHAR_HIDDEN, 2).to(DEVICE)
opt_d   = torch.optim.Adam(model_d.parameters(), lr=LR)

In [24]:
# Part D — Обучаем символьную GRU модель
print('Обучение Char-Level GRU (Part D)...')
for epoch in range(EPOCHS_D):
    tr_loss, tr_acc = train_epoch(model_d, train_loader_d, opt_d, criterion)
    val_acc_d, _, _ = evaluate(model_d, val_loader_d)
    print(f'  Эпоха {epoch+1}/{EPOCHS_D} | Loss: {tr_loss:.4f} | Train: {tr_acc:.4f} | Val: {val_acc_d:.4f}')

Обучение Char-Level GRU (Part D)...
  Эпоха 1/5 | Loss: 0.6943 | Train: 0.5097 | Val: 0.5006
  Эпоха 2/5 | Loss: 0.6910 | Train: 0.5249 | Val: 0.5172
  Эпоха 3/5 | Loss: 0.6889 | Train: 0.5291 | Val: 0.5218
  Эпоха 4/5 | Loss: 0.6858 | Train: 0.5485 | Val: 0.5470
  Эпоха 5/5 | Loss: 0.6774 | Train: 0.5675 | Val: 0.5860


In [25]:
# Part D — Итоговые метрики и примеры, где char модель отличается от word модели
train_acc_d, _,            _            = evaluate(model_d, train_loader_d)
val_acc_d,   _,            _            = evaluate(model_d, val_loader_d)
test_acc_d,  test_preds_d, test_true_d  = evaluate(model_d, test_loader_d)

print(f'Train Accuracy : {train_acc_d:.4f}')
print(f'Val   Accuracy : {val_acc_d:.4f}')
print(f'Test  Accuracy : {test_acc_d:.4f}')

# Находим примеры, где два классификатора дают разные ответы
diff_idxs = [i for i,(pw,pc) in enumerate(zip(test_preds_b, test_preds_d)) if pw != pc][:3]

print('\n3 примера где char модель предсказала ИНАЧЕ чем word модель:')
lbl = lambda x: 'POS' if x == 1 else 'NEG'
if diff_idxs:
    for i in diff_idxs:
        print(f'\n  Текст    : {test_texts[i][:250]}...')
        print(f'  Правда   : {lbl(test_true_d[i])}')
        print(f'  Word GRU : {lbl(test_preds_b[i])}')
        print(f'  Char GRU : {lbl(test_preds_d[i])}')
else:
    print('  Обе модели согласны на всех тестовых примерах.')

Train Accuracy : 0.6127
Val   Accuracy : 0.5860
Test  Accuracy : 0.5780

3 примера где char модель предсказала ИНАЧЕ чем word модель:

  Текст    : Worth the entertainment value of a rental, especially if you like action movies. This one features the usual car chases, fights with the great Van Damme kick style, shooting battles with the 40 shell load shotgun, and even terrorist style bombs. All ...
  Правда   : NEG
  Word GRU : NEG
  Char GRU : POS

  Текст    : STAR RATING: ***** Saturday Night **** Friday Night *** Friday Morning ** Sunday Night * Monday Morning <br /><br />Former New Orleans homicide cop Jack Robideaux (Jean Claude Van Damme) is re-assigned to Columbus, a small but violent town in Mexico ...
  Правда   : NEG
  Word GRU : NEG
  Char GRU : POS

  Текст    : Isaac Florentine has made some of the best western Martial Arts action movies ever produced. In particular US Seals 2, Cold Harvest, Special Forces and Undisputed 2 are all action classics. You can tell Isaac has a

---
## Part E: Compare All Approaches

In [26]:
# Part E — Итоговая сравнительная таблица всех моделей
print(f"""
+-----------------------------+------------+-----------+--------+------+-------+-------+-------+
| Модель / Представление      | Преп.      | OOV       | Vocab  | MaxL | Train | Val   | Test  |
+-----------------------------+------------+-----------+--------+------+-------+-------+-------+
| Word IDs + GRU              | Simple     | <UNK>     |{VOCAB_SIZE:>7} | {MAX_SEQ_LEN:>4} | {train_acc_b:.3f} | {val_acc_b:.3f} | {test_acc_b:.3f} |
| Gensim W2V + LogReg         | Simple     | {best_oov:<9} |   100d |  N/A | {train_acc_c:.3f} | {val_acc_c:.3f} | {test_acc_c:.3f} |
| Char GRU                    | Сырые симв | <UNK>     |{CHAR_VOCAB_SIZE:>7} | {MAX_CHAR_LEN:>4} | {train_acc_d:.3f} | {val_acc_d:.3f} | {test_acc_d:.3f} |
+-----------------------------+------------+-----------+--------+------+-------+-------+-------+
""")


+-----------------------------+------------+-----------+--------+------+-------+-------+-------+
| Модель / Представление      | Преп.      | OOV       | Vocab  | MaxL | Train | Val   | Test  |
+-----------------------------+------------+-----------+--------+------+-------+-------+-------+
| Word IDs + GRU              | Simple     | <UNK>     |  34479 |  256 | 0.994 | 0.852 | 0.839 |
| Gensim W2V + LogReg         | Simple     | skip      |   100d |  N/A | 0.836 | 0.833 | 0.831 |
| Char GRU                    | Сырые симв | <UNK>     |    170 |  512 | 0.613 | 0.586 | 0.578 |
+-----------------------------+------------+-----------+--------+------+-------+-------+-------+



### Part E — Ответы

**1. Какое представление сработало лучше?**  
Word IDs + GRU — модель учит контекстно-зависимые представления через рекуррентный слой, в отличие от статичного mean pooling в Gensim.

**2. Какой выбор препроцессинга повлиял больше всего?**  
Lowercase и удаление HTML-тегов. Удаление стоп-слов — рискованный шаг, так как слова *«not»*, *«never»* меняют смысл.

**3. Какая OOV стратегия сработала лучше?**  
Zero vector и skip дали схожие результаты. Avg_known теоретически надёжнее при большом количестве OOV.

**4. Какую модель выбрать при опечатках и редких словах?**  
Символьную модель (Part D) — она никогда не даёт `<UNK>` для целого слова, любое слово раскладывается на символы.

---
## Follow-Up Questions

**1. Разница между токеном, элементом словаря и вектором эмбеддинга?**  
**Токен** — строка после токенизации (`"good"`). **Элемент словаря** — уникальный токен с целочисленным индексом (`"good" → 42`). **Вектор эмбеддинга** — плотный вещественный вектор (например, 128 чисел), который представляет токен в непрерывном пространстве.

**2. Почему словарь строится только из тренировочных данных?**  
Чтобы избежать утечки данных. На практике модель никогда не видит тест до инференса — если включить тестовый словарь, метрики будут завышены.

**3. Что представляет `<UNK>`?**  
Специальный токен для слов, которых нет в словаре — слишком редких или не встречавшихся при обучении. Модель учит один общий вектор для всех OOV слов.

**4. Почему пропуск OOV слов опасен для сентимент-анализа?**  
Редкие слова часто несут сильный сентимент: `"abysmal"`, `"breathtaking"`, сленг. Пропустив `"abysmal"` в «This film was utterly abysmal», модель теряет главный негативный сигнал.

**5. Почему символьная модель лучше справляется с неизвестными словами?**  
Символы почти всегда есть в словаре. Любое слово — даже опечатка или неологизм — может быть хотя бы частично представлено через последовательность символов. Word-модель сводит все OOV к одному вектору `<UNK>`.

**6. Что теряется при представлении отзыва средним вектором слов?**  
Порядок слов исчезает полностью: «not good» и «good, not bad» могут дать похожий mean-вектор несмотря на противоположный смысл. Редкие, но важные слова растворяются в среднем.

**7. Почему удаление стоп-слов иногда вредит сентимент-анализу?**  
Стоп-слова `"not"`, `"no"`, `"never"` — ключевые модификаторы сентимента. «I do **not** like this» → после удаления → «I like this» — смысл полностью обратный.

**8. Какой подход был наиболее чувствителен к препроцессингу?**  
Word-level модели (B и C) — их словарь строится прямо из токенизации. Символьная модель (D) наименее чувствительна — она читает сырые символы, минуя токенизацию.